In [1]:
import subprocess
import os

result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value

In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='TRUE'
# 1. 设置镜像（必须在导入 datasets 之前生效，或者确保环境已设置）
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
import time
import datasets
from datasets import load_dataset
from requests.exceptions import ChunkedEncodingError, ConnectionError, ReadTimeout


# 2. 配置下载参数
config = datasets.DownloadConfig(resume_download=True, max_retries=100)

# 3. 定义下载函数，包裹在循环中
def download_with_retry():
    max_retries_external = 50  # 外部强行重试次数
    attempt = 0
    
    while attempt < max_retries_external:
        try:
            print(f">>> 开始尝试下载 (第 {attempt + 1} 次)...")
            
            ds = load_dataset(
                "agupte/MedVQA", 
                cache_dir="../../datasets/cache/medvqa", 
                token=True, 
                download_config=config,
                # 某些情况下，多进程会导致断流更频繁，如果还报错，可以尝试把 num_proc 设为 1
                # num_proc=1 
            )
            
            print(">>> ✅ 下载并加载成功！")
            return ds
            
        except (ChunkedEncodingError, ConnectionError, ReadTimeout, Exception) as e:
            # 捕捉所有网络中断相关的错误
            print(f">>> ❌ 发生错误: {type(e).__name__}")
            print(f">>> 错误详情: {e}")
            print(f">>> ⚠️ 正在休眠 5 秒后准备断点续传...")
            time.sleep(5)
            attempt += 1
            
    raise RuntimeError("达到最大重试次数，下载失败。请检查网络或硬盘空间。")

# 4. 执行下载
if __name__ == "__main__":
    ds = download_with_retry()

>>> 开始尝试下载 (第 1 次)...


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/15.2M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/2.18M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/635 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/159 [00:00<?, ? examples/s]

>>> ✅ 下载并加载成功！


In [3]:
print(ds['train'][:1])

{'ids': ['0'], 'image_names': ['synpic26764.jpg'], 'images': [<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1024x1192 at 0x7F527E95AFF0>], 'questions': ['Is this film taken in a PA modality?'], 'answers': ['Yes']}


In [4]:
import os
import json
from datasets import load_dataset
from tqdm import tqdm

# ================= 配置区域 =================
# 1. 这里填你刚才下载时的 cache_dir
# 脚本会直接从这个目录读取缓存，不需要重新下载
CACHE_DIR = "../../datasets/cache/medvqa" 
DATASET_NAME = "agupte/MedVQA"

# 2. 输出配置（从 training/notebooks 目录运行）
OUTPUT_IMG_DIR = "../../datasets/processed/llamafactory/medvqa-images"
OUTPUT_JSON_FILE = "../../datasets/processed/llamafactory/metadata/medvqa_data.json"

# ===========================================

def process_data():
    # 1. 确保输出目录存在
    if not os.path.exists(OUTPUT_IMG_DIR):
        os.makedirs(OUTPUT_IMG_DIR)
        print(f"已创建图片目录: {OUTPUT_IMG_DIR}")
    
    # 2. 加载数据集 (利用你刚才下载的缓存)
    print(f"正在从缓存 {CACHE_DIR} 加载数据集...")
    try:
        # 注意：这里必须保持和下载时一致的参数，它才会命中缓存
        ds = load_dataset(DATASET_NAME, cache_dir=CACHE_DIR, split="train")
    except Exception as e:
        print(f"加载失败，请检查路径: {e}")
        return

    all_data = []
    print(f"开始处理 {len(ds)} 条数据...")

    # 3. 遍历所有数据
    for i, item in tqdm(enumerate(ds), total=len(ds)):
        try:
            # --- A. 获取图片对象 ---
            image = item["images"] 
            
            # --- B. 确定文件名 ---
            # 优先使用数据集里的原始文件名，防止重名加上索引
            if item.get("image_names"):
                raw_name = item["image_names"]
                # 有些文件名可能带路径，只取文件名部分
                filename = os.path.basename(raw_name) 
            else:
                filename = f"med_{i:05d}.jpg"
            
            # --- C. 保存图片 ---
            save_path = os.path.join(OUTPUT_IMG_DIR, filename)
            
            # 强制转 RGB (这一步很关键，防止 PNG/RGBA 报错)
            if image.mode != "RGB":
                image = image.convert("RGB")
            image.save(save_path)

            # --- D. 构建 JSON 数据 ---
            # 图片路径相对于 LlamaFactory 的 dataset_dir
            json_img_path = f"medvqa_images/{filename}"
            
            question = item["questions"]
            answer = item["answers"]

            all_data.append({
                "messages": [
                    {
                        "content": f"<image>{question}", 
                        "role": "user"
                    },
                    {
                        "content": answer,
                        "role": "assistant"
                    }
                ],
                "images": [
                    json_img_path
                ]
            })

        except Exception as e:
            print(f"跳过第 {i} 条数据: {e}")
            continue

    # 4. 保存 JSON 到 data 根目录
    with open(OUTPUT_JSON_FILE, "w", encoding='utf-8') as f:
        json.dump(all_data, f, ensure_ascii=False, indent=2)

    print(f"\n✅ 处理完成！")
    print(f"图片位置: {OUTPUT_IMG_DIR}")
    print(f"JSON位置: {OUTPUT_JSON_FILE}")
    print(f"有效数据: {len(all_data)} 条")

if __name__ == "__main__":
    process_data()

已创建图片目录: ../../datasets/processed/llamafactory/medvqa-images
正在从缓存 ../../datasets/cache/medvqa 加载数据集...
开始处理 635 条数据...


100%|██████████| 635/635 [00:17<00:00, 35.71it/s]



✅ 处理完成！
图片位置: ../../datasets/processed/llamafactory/medvqa-images
JSON位置: ../../datasets/processed/llamafactory/metadata/medvqa_data.json
有效数据: 635 条


In [9]:
save_images_and_json(ds['train'])

Processing_Image: 100%|██████████| 8000/8000 [00:29<00:00, 274.92image/s]
